In [122]:
## Use Case

##Problem Statement
# Planning a trip often requires users to search across multiple sources to decide destinations, duration, places to visit, transport options, stay preferences, estimated budget, and daily schedules. This process is time-consuming and fragmented. There is a need for an intelligent system that can take user inputs such as source location, destination, duration, budget, travel style, and interests, and generate a structured, personalized travel itinerary.
# Objective
# To build an AI-powered travel agent that generates customized, day-wise itineraries using Groq LLM and provides an interactive interface using Gradio.

## Functional Requirements
# - Accept user inputs (source, destination, days, budget, style, interests)
# - Generate trip summary
# - Provide day-wise itinerary
# - Recommend attractions and food
# - Suggest travel tips and budget guidance
# - Display output via a web interface


In [123]:
!pip install --upgrade gradio langchain===1.2.13 langchain-core===1.2.23 langchain_groq===1.1.2 google-search-results===2.4.2 langchain-community===0.4.1

In [124]:
# g_K_key
import gradio as gr
from langchain_core.prompts import PromptTemplate
from langchain_groq import ChatGroq
from google.colab import userdata
from langchain_core.output_parsers import StrOutputParser
from langchain_community.utilities import SerpAPIWrapper
from langchain.agents import create_agent
from langchain.tools import tool, ToolRuntime
from langchain.agents.middleware import ToolCallLimitMiddleware
from langchain_groq import ChatGroq
import os
import time

In [125]:


def get_prompt_template():
  prompt_template = PromptTemplate.from_template(
        """
You are an expert AI trip planner.

Using the following inputs, generate a realistic, detailed, day-by-day travel itinerary.

User Inputs:
- Source: {source}
- Destination: {destination}
- Days: {days}
- Budget: {budget}
- Travel Style: {style}
- Interests: {interests}

Requirements:
1. Create a day-by-day itinerary
2. Stay within budget
3. Match the travel style
4. Include hotel, transport, food, and activities
5. Add useful local tips
6. Keep the plan realistic and efficient

Return only the final itinerary in clean markdown.
"""
    )
  return prompt_template

def get_llm(stream_response=False):
  os.environ["GROQ_API_KEY"] =userdata.get('LLM_API_KEY')
  return ChatGroq(
      model="openai/gpt-oss-120b",
      temperature=0,
      max_completion_tokens=8192,
      top_p=1,
      reasoning_effort="medium",
      stream=stream_response,
      stop=None
  )

##### Using Search Agent

def get_search():
    os.environ["SERPAPI_API_KEY"] =userdata.get('SERP_API_KEY')
    return SerpAPIWrapper()

@tool
def search_flights(query: str, runtime:ToolRuntime) -> str:
    """Search live web results for flight options, airlines, stops, duration, and prices."""
    start = time.time()
    writer = runtime.stream_writer
    writer({"tool": "flights", "status": "running", "time": None})

    search = get_search()
    enriched_query = (
        f"{query}. Find top flight options with airline, airports, business/economy class "
        f"where available, approximate fare, duration, and number of stops."
    )
    result = search.run(enriched_query)

    duration = round(time.time() - start, 2)
    writer({"tool": "flights", "status": "done", "time": duration})
    return result

@tool
def search_hotels(query: str, runtime: ToolRuntime) -> str:
    """Search live web results for hotels, nightly price, amenities, and neighborhood."""
    start = time.time()
    writer = runtime.stream_writer
    writer({"tool": "hotels", "status": "running", "time": None})

    search = get_search()
    enriched_query = (
        f"{query}. Find hotel options with nightly price, star rating, neighborhood, "
        f"amenities, and notable pros."
    )
    result = search.run(enriched_query)

    writer("Finished hotel search.")
    duration = round(time.time() - start, 2)
    writer({"tool": "hotels", "status": "done", "time": duration})
    return result


@tool
def search_activities(query: str, runtime: ToolRuntime) -> str:
    """Search live web results for attractions, restaurants, local experiences, and tips."""
    start = time.time()
    writer = runtime.stream_writer
    writer({"tool": "activities", "status": "running", "time": None})

    search = get_search()
    enriched_query = (
        f"{query}. Find top attractions, restaurants, local experiences, opening hours "
        f"if available, and practical visitor tips."
    )
    result = search.run(enriched_query)
    duration = round(time.time() - start, 2)
    writer({"tool": "activities", "status": "done", "time": duration})
    return result
# =========================
# Helpers
# =========================
def _extract_text_from_message_content(content) -> str:
    if isinstance(content, str):
        return content.strip()

    if isinstance(content, list):
        parts = []
        for item in content:
            if isinstance(item, dict) and item.get("type") == "text":
                parts.append(item.get("text", ""))
        return "\n".join(parts).strip()

    return str(content).strip()


def _friendly_step_name(step_name: str) -> str:
    mapping = {
        "model": "Building the itinerary...",
        "tools": "Using travel search tools...",
        "search_flights": "Searching flight options...",
        "search_hotels": "Checking hotels...",
        "search_activities": "Finding attractions and dining...",
    }
    return mapping.get(step_name, f"Working on: {step_name}")

def format_status(status_map):
    def row(label, key):
        item = status_map[key]
        state = item["status"]
        elapsed = item["time"]

        icon = {
            "pending": "⚪",
            "running": "⏳",
            "done": "✅",
            "error": "❌",
        }.get(state, "⚪")

        time_text = f"{elapsed}s" if elapsed is not None else "--"
        return f"{icon} {label} - {state.upper()} - {time_text}"

    return "\n".join([
        row("Flight Search", "flights"),
        row("Hotel Search", "hotels"),
        row("Activity Search", "activities"),
    ])

def build_trip_agent():
    llm = get_llm()

    system_prompt = """
# You are an expert AI trip planner.

# Your job is to create a realistic, useful, premium-quality itinerary using live web search.
# You have tools for flights, hotels, and activities.

# Rules:
# 1. Always search before recommending flights, hotels, or activities.
# 2. For flights, identify the top 3 options only.
# 3. Keep recommendations aligned to the user's budget, travel style, and interests.
# 4. Be practical: avoid overpacked schedules and unrealistic transit assumptions.
# 5. If exact live prices vary, present them as approximate and say so.
# 6. Return clean markdown only.
# 7. In the final answer, include:
#    - A short trip overview
#    - Top 3 flight options in a table
#    - Recommended hotel options
#    - Day-by-day itinerary
#    - Estimated budget summary
#    - Key travel tips
# 8. Do not expose tool-call traces or intermediate reasoning.
"""
    system_prompt= """
    You are an expert AI trip planner.

    Rules:
    1. You may use the flight search tool only.
    2. Use the flight search tool at most once.
    3. Do not search hotels or activities with tools.
    4. For hotels and activities, use your general planning ability and clearly avoid pretending you fetched live hotel/activity data.
    5. Return clean markdown only.
    6. Include:
      - Short trip overview
      - Top 3 flight options
      - Hotel recommendations
      - Activity recommendations
      - Day-by-day itinerary
      - Budget summary
    """

    agent = create_agent(
        model=llm,
        tools=[search_flights],#, search_hotels, search_activities],
        system_prompt=system_prompt,
        middleware=[
            ToolCallLimitMiddleware(tool_name="search_flights", run_limit=1),
            ToolCallLimitMiddleware(run_limit=1),
        ],
    )
    return agent

In [126]:
def generate_trip_plan(source, destination, days, budget, style, interests, stream=True):

    prompt_input = {
        "source": source.strip(),
        "destination": destination.strip(),
        "days": int(days),
        "budget": budget.strip(),
        "style": style.strip(),
        "interests": interests.strip(),
    }

    prompt_template = get_prompt_template()
    llm = get_llm(stream)

    # Initial UI state
    yield (
        "",  # output_md
        "Generating itinerary...",  # status
        "",  # copy_state
        gr.Accordion(open=True, visible=True),   # show progress
        gr.Column(visible=False),                # hide result
        gr.Button(interactive=False),            # disable copy
    )

    if stream:
        partial_text = ""
        show_progress = True
        try:
            for chunk in llm.stream(prompt_template.format(**prompt_input)):
                text = getattr(chunk, "content", "")
                if text:
                    partial_text += text

                    yield (
                      partial_text,                          # show streamed markdown
                      "Generating itinerary...",
                      partial_text,
                      gr.Accordion(open=True, visible=True), # optional: keep status visible
                      gr.Column(visible=True),               # show result panel now
                      gr.Button(interactive=False),
                  )

            # FINAL OUTPUT
            # yield (
            #     partial_text,
            #     "Done.",
            #     partial_text,
            #     gr.Accordion(open=False, visible=False),
            #     gr.Column(visible=True),
            #     gr.Button(interactive=True),
            # )
            yield (
                partial_text,
                "Done.",
                partial_text,
                gr.Accordion(open=False, visible=False),
                gr.Column(visible=True),
                gr.Button(interactive=True),
            )
            show_progress=False

        except Exception as e:
            err = f"Error: {str(e)}"
            yield (
                "",
                err,
                "",
                gr.Accordion(open=True, visible=True),
                gr.Column(visible=False),
                gr.Button(interactive=False),
            )

    else:
        output_parser = StrOutputParser()
        chain = prompt_template | llm | output_parser
        result = chain.invoke(prompt_input)

        yield (
            result,
            "Done.",
            result,
            gr.Accordion(open=False, visible=False),
            gr.Column(visible=True),
            gr.Button(interactive=True),
        )

In [127]:


# =========================
# Main generator
# Outputs:
# 1) final markdown text
# 2) progress text
# 3) copy_state
# 4) progress accordion update
# 5) result column update
# 6) copy button update
# =========================
def generate_trip_plan_with_agent(
    source: str,
    destination: str,
    days,
    budget: str,
    style: str,
    interests: str,
):
    source = (source or "").strip()
    destination = (destination or "").strip()
    budget = (budget or "").strip()
    style = (style or "").strip()
    interests = (interests or "").strip()

    if not source or not destination or not budget or not style or not interests:
        msg = "Please fill in all fields."
        yield (
            "",  # output_md
            msg,  # status_box
            "",  # copy_state
            gr.Accordion(open=True, visible=True),   # progress panel stays visible
            gr.Column(visible=False),                # final result stays hidden
            gr.Button(interactive=False),            # copy disabled
        )
        return

    try:
        days = int(days)
    except Exception:
        msg = "Days must be a whole number."
        yield (
            "",
            msg,
            "",
            gr.Accordion(open=True, visible=True),
            gr.Column(visible=False),
            gr.Button(interactive=False),
        )
        return

    agent = build_trip_agent()


    user_request = f"""
Plan a trip with these inputs:

- Source: {source}
- Destination: {destination}
- Days: {days}
- Budget: {budget}
- Travel Style: {style}
- Interests: {interests}

Requirements:
1. Search the web and show the top 3 flight options from {source} to {destination}
2. Recommend hotel/stay options that match the budget and travel style
3. Suggest activities, attractions, restaurants, and local experiences
4. Build a realistic day-by-day itinerary
5. Provide a concise budget summary

Output:
- Markdown only
- Use headings
- Include a flight table
- Include a budget summary table
"""
    user_request = user_request = f"""
Plan a trip with these inputs:

- Source: {source}
- Destination: {destination}
- Days: {days}
- Budget: {budget}
- Travel Style: {style}
- Interests: {interests}

Requirements:
1. Use the flight search tool once to find top 3 flight options
2. Do not use tools for hotels or activities
3. For hotels and activities, give practical recommendations without claiming live search data
4. Build a realistic day-by-day itinerary
5. Provide a concise budget summary

Output:
- Markdown only
- Use headings
- Include a flight table
- Include a budget summary table
"""

    status = {
    "flights": "Flight Search - pending",
    "hotels": "Hotel Search - skipped",
    "activities": "Activity Search - skipped"
}

    status_map = {
    "flights": {"status": "pending", "time": None},
    "hotels": {"status": "SKIPPED", "time": None},
    "activities": {"status": "SKIPPED", "time": None},
}
    final_text = ""
    status_lines = []

  #     # Initial UI state while running
  #     yield (
  #         "",  # no ugly partial markdown
  #         "\n".join(status_lines),
  #         "",
  #         gr.Accordion(open=True, visible=True),
  #         gr.Column(visible=False),
  #         gr.Button(interactive=False),
  #     )

    try:
        for mode, chunk in agent.stream(
            {"messages": [{"role": "user", "content": user_request}]},
            stream_mode=["updates", "custom"],
            #stream_mode=["custom"],
        ):
            if mode == "custom":
              if isinstance(chunk, dict):
                  tool_name = chunk.get("tool")
                  tool_status = chunk.get("status")
                  tool_time = chunk.get("time")

                  if tool_name in status_map:
                      status_map[tool_name]["status"] = tool_status
                      status_map[tool_name]["time"] = tool_time

              yield (
                  "",                              # keep main markdown hidden while searching
                  format_status(status_map),       # only 3 clean lines
                  "",
                  gr.Accordion(open=True, visible=True),
                  gr.Column(visible=False),
                  gr.Button(interactive=False),
              )

                # status_text = "\n".join(status.values())
                # if isinstance(chunk, str) and chunk.strip():
                #     if not status_lines or status_lines[-1] != chunk.strip():
                #         status_lines.append(chunk.strip())

            elif mode == "updates":
              if isinstance(chunk, dict):
                  for step_name, step_data in chunk.items():
                      if not isinstance(step_data, dict):
                          continue

                      messages = step_data.get("messages", [])
                      if not messages:
                          continue

                      last_message = messages[-1]
                      content = getattr(last_message, "content", "")
                      text = _extract_text_from_message_content(content)

                      if (
                          text
                          and len(text) > 300
                          and ("## " in text or "# " in text or "|" in text)
                      ):
                          final_text = text


        if final_text:
            status_lines.append("Done.")
            yield (
                final_text,                           # render final markdown now
                "\n".join(status_lines),
                final_text,                           # copy source
                gr.Accordion(open=False, visible=False),  # hide/collapse progress
                gr.Column(visible=True),             # show result panel
                gr.Button(interactive=True),         # enable copy
            )
        else:
            msg = "No final itinerary was generated."
            yield (
                "",
                "\n".join(status_lines + [msg]),
                "",
                gr.Accordion(open=True, visible=True),
                gr.Column(visible=False),
                gr.Button(interactive=False),
            )

    except Exception as e:
        err = f"Error: {str(e)}"
        yield (
            "",
            "\n".join(status_lines + [err]),
            "",
            gr.Accordion(open=True, visible=True),
            gr.Column(visible=False),
            gr.Button(interactive=False),
        )



In [128]:
generate_trip_plan_with_agent("Chicago","KOlkata",10,"$12000","Luxury", "food,fun,chill, relax, toddler friendly")

<generator object generate_trip_plan_with_agent at 0x7bdb7cd56840>

In [129]:
def generate_trip_plan_router(mode, source, destination, days, budget, style, interests):
    if mode == "Agent":
        yield from generate_trip_plan_with_agent(
            source, destination, days, budget, style, interests
        )
    else:
        yield from generate_trip_plan(
            source, destination, days, budget, style, interests
        )

In [130]:
import gradio as gr

COPY_JS = """
async (text) => {
    if (!text) return;
    await navigator.clipboard.writeText(text);
}
"""

CUSTOM_CSS = """
.main-wrap {
    max-width: 980px;
    margin: 0 auto;
    padding-top: 12px;
}

.hero h1 {
    margin-bottom: 0.25rem;
}

.hero p {
    opacity: 0.8;
    margin-top: 0;
    margin-bottom: 1.25rem;
}

.card {
    border: 1px solid rgba(255,255,255,0.08);
    border-radius: 16px;
    padding: 14px;
    background: rgba(255,255,255,0.02);
}

.copy-btn button {
    min-width: 84px !important;
}

.generate-btn button {
    min-height: 44px !important;
    font-weight: 600 !important;
}

#trip-output {
    border: 1px solid rgba(255,255,255,0.08);
    border-radius: 16px;
    padding: 18px;
    background: rgba(255,255,255,0.02);
}
"""


def generate_trip_plan_router(mode, source, destination, days, budget, style, interests):
    if mode == "Agent":
        yield from generate_trip_plan_with_agent(
            source, destination, days, budget, style, interests
        )
    else:
        yield from generate_trip_plan(
            source, destination, days, budget, style, interests
        )


with gr.Blocks(css=CUSTOM_CSS, theme=gr.themes.Soft(), title="AI Trip Planner") as demo:
    copy_state = gr.State("")

    with gr.Column(elem_classes="main-wrap"):
        gr.Markdown(
            """
<div class="hero">
  <h1>AI Trip Planner</h1>
  <p>Build a personalized itinerary based on budget, style, and interests.</p>
</div>
"""
        )

        with gr.Column(elem_classes="card"):
            mode = gr.Radio(
                choices=["No Agent", "Agent"],
                value="No Agent",
                label="Planning Mode",
            )

            with gr.Row():
                source = gr.Textbox(label="Source", placeholder="e.g. Chicago")
                destination = gr.Textbox(label="Destination", placeholder="e.g. Tokyo")

            with gr.Row():
                days = gr.Number(label="Days", value=5, precision=0)
                budget = gr.Textbox(label="Budget", placeholder="e.g. $2000")

            with gr.Row():
                style = gr.Dropdown(
                    label="Travel Style",
                    choices=["Budget", "Luxury", "Adventure", "Family", "Romantic", "Backpacking"],
                    value="Budget",
                )
                interests = gr.Textbox(
                    label="Interests",
                    placeholder="e.g. food, culture, temples, hiking",
                )

            generate_btn = gr.Button(
                "Generate Itinerary",
                variant="primary",
                elem_classes="generate-btn",
            )

        with gr.Accordion("Research in progress", open=True, visible=False) as progress_panel:
            status_box = gr.Textbox(
                label="Status",
                lines=8,
                interactive=False,
                autoscroll=True,
            )

        with gr.Column(visible=False) as result_panel:
            with gr.Row():
                gr.Markdown("### Your itinerary")
                copy_btn = gr.Button(
                    "Copy",
                    elem_classes="copy-btn",
                    interactive=False,
                )

            output_md = gr.Markdown(elem_id="trip-output")

    generate_btn.click(
        fn=generate_trip_plan_router,
        inputs=[mode, source, destination, days, budget, style, interests],
        outputs=[output_md, status_box, copy_state, progress_panel, result_panel, copy_btn],
    )

    copy_btn.click(
        fn=None,
        inputs=copy_state,
        outputs=None,
        js=COPY_JS,
    )

demo.queue()
demo.launch()

/tmp/ipykernel_42686/3110592091.py:63: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme, css. Please pass these parameters to launch() instead.
  with gr.Blocks(css=CUSTOM_CSS, theme=gr.themes.Soft(), title="AI Trip Planner") as demo:


It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://38f5ed953c04319f49.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [131]:
def test_router(mode="No Agent"):
    gen = generate_trip_plan_router(
        mode=mode,
        source="Chicago",
        destination="Kolkata",
        days=14,
        budget="$8000",
        style="Family",
        interests="Fun, toddler and vegetarian friendly"
    )

    for i, step in enumerate(gen, 1):
        output_md, status_box, copy_state, progress_panel, result_panel, copy_btn = step

        print(f"\n=== STEP {i} ===")
        print("Status:")
        print(status_box)
        print("\nOutput preview:")
        print((output_md or "")[:500])

In [132]:
# test_router("No Agent")
test_router("Agent")

/usr/local/lib/python3.12/dist-packages/pydantic/main.py:250: UserWarning: WARNING! max_completion_tokens is not default parameter.
                    max_completion_tokens was transferred to model_kwargs.
                    Please confirm that max_completion_tokens is what you intended.
  validated_self = self.__pydantic_validator__.validate_python(data, self_instance=self)
/usr/local/lib/python3.12/dist-packages/pydantic/main.py:250: UserWarning: WARNING! top_p is not default parameter.
                    top_p was transferred to model_kwargs.
                    Please confirm that top_p is what you intended.
  validated_self = self.__pydantic_validator__.validate_python(data, self_instance=self)
/usr/local/lib/python3.12/dist-packages/pydantic/main.py:250: UserWarning: WARNING! stream is not default parameter.
                    stream was transferred to model_kwargs.
                    Please confirm that stream is what you intended.
  validated_self = self.__pydantic_validat


=== STEP 1 ===
Status:
⏳ Flight Search - RUNNING - --
⚪ Hotel Search - SKIPPED - --
⚪ Activity Search - SKIPPED - --

Output preview:


=== STEP 2 ===
Status:
✅ Flight Search - DONE - 0.5s
⚪ Hotel Search - SKIPPED - --
⚪ Activity Search - SKIPPED - --

Output preview:


=== STEP 3 ===
Status:
Done.

Output preview:
# 14‑Day Family Trip: Chicago ➜ Kolkata  
**Travel dates (example):** 15 May 2024 – 29 May 2024  
**Travel style:** Family‑friendly, toddler‑ and vegetarian‑oriented  
**Total budget:** **US $8,000**  

---

## 1. Flight Options  *(search performed once)*  

| # | Airline & Route | Stops | Total travel time* | Approx. round‑trip price (per adult) | Notes |
|---|-----------------|-------|--------------------|--------------------------------------|-------|
| 1 | **Air India** – ORD → DEL → CCU | 1
